In [30]:
import pandas as pd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mat
import datetime
import time
from tqdm import tqdm

#设置中文字体
mat.rcParams['font.family']='SimHei'
mat.rcParams['font.sans-serif']='SimHei'

import sys
sys.path.append('../plot/')
import myplot
sys.path.append('../utils/')
import utils

#忽略警告
import warnings
warnings.filterwarnings('ignore')

from importlib import reload

In [5]:
df_read = pd.read_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dateLabeled_advance.csv")
df_read

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-43,2022-07-05 23:49:47,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
2,22,-43,2022-07-05 23:50:00,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
3,22,-35,2022-07-05 23:50:00,"7.5-20,178,229,137,47,142","20,178,229,137,47,142",7.5
4,22,-42,2022-07-05 23:50:13,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
...,...,...,...,...,...,...
1154356,32,-34,2022-08-07 23:51:18,"8.7-156,95,90,12,124,73","156,95,90,12,124,73",8.7
1154357,32,-42,2022-08-07 23:51:44,"8.7-124,161,119,201,50,40","124,161,119,201,50,40",8.7
1154358,32,-27,2022-08-07 23:51:57,"8.7-156,95,90,12,124,73","156,95,90,12,124,73",8.7
1154359,32,-36,2022-08-07 23:52:49,"8.7-124,161,119,201,50,40","124,161,119,201,50,40",8.7


In [59]:
df_read[df_read.oriMac == '32,50,51,210,96,61'].m.unique()

array(['7.6-32,50,51,210,96,61', '7.7-32,50,51,210,96,61',
       '7.8-32,50,51,210,96,61', '7.9-32,50,51,210,96,61',
       '7.10-32,50,51,210,96,61', '7.11-32,50,51,210,96,61',
       '7.12-32,50,51,210,96,61', '7.13-32,50,51,210,96,61',
       '7.14-32,50,51,210,96,61', '7.15-32,50,51,210,96,61',
       '7.18-32,50,51,210,96,61', '7.19-32,50,51,210,96,61',
       '7.20-32,50,51,210,96,61', '7.22-32,50,51,210,96,61',
       '7.24-32,50,51,210,96,61', '7.25-32,50,51,210,96,61',
       '7.26-32,50,51,210,96,61', '7.27-32,50,51,210,96,61',
       '7.28-32,50,51,210,96,61', '7.29-32,50,51,210,96,61',
       '7.30-32,50,51,210,96,61', '7.31-32,50,51,210,96,61',
       '8.1-32,50,51,210,96,61', '8.2-32,50,51,210,96,61',
       '8.3-32,50,51,210,96,61', '8.4-32,50,51,210,96,61',
       '8.5-32,50,51,210,96,61', '8.6-32,50,51,210,96,61',
       '8.7-32,50,51,210,96,61'], dtype=object)

In [14]:
#get mac list
df = df_read.copy()
df.t = pd.to_datetime(df.t)
mac_list = df.oriMac.unique()
print('mac num : ' , len(mac_list))

mac num :  312


In [38]:
#if a MAC stay for more than 3 days and at least 10 hours for each day, this mac will be considered as resident's MAC


def DivideId(df_now):
    days = df_now.calender.unique()
    count = 0
    isPassing = 0
    for i in range(len(days)):
        df_one = df_now[df_now.calender == days[i]]
        time = df_one.iloc[len(df_one)-1].t - df_one.iloc[0].t
        h = time.components.hours
        if h>10:
            count += 1
        elif h<0.5:
            isPassing += 1
    if isPassing > 5:
        return 'pass'
    elif count < 4:
        return 'tourist'
    else:
        return 'resident'
    

tourist_list = []
resident_list = []
pass_list = []

for mac in tqdm(mac_list,desc="Dividing:"):
    df_now = df[df.oriMac == mac]
    df_now.sort_values(by='t')
    result = DivideId(df_now)
    if result == 'pass':
        pass_list.append(mac)
    elif result == 'tourist':
        tourist_list.append(mac)
    else:
        resident_list.append(mac)


Dividing:: 100%|██████████| 312/312 [00:24<00:00, 12.95it/s]


In [40]:
print('pass count:',len(pass_list))
print('tourists count:',len(tourist_list))
print('residents count:',len(resident_list))

pass count: 0
tourists count: 242
residents count: 70


In [41]:
#add id to main df
def AddId(mac):
    if mac in tourist_list:
        return 't'
    if mac in resident_list:
        return 'r'
    if mac in pass_list:
        return 'p'
df['id'] = df.oriMac.apply(AddId)
df.head()

,a,r,t,m,oriMac,calender,id
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,r
1,22,-43,2022-07-05 23:49:47,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,r
2,22,-43,2022-07-05 23:50:00,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,r
3,22,-35,2022-07-05 23:50:00,"7.5-20,178,229,137,47,142","20,178,229,137,47,142",7.5,r
4,22,-42,2022-07-05 23:50:13,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,r


In [42]:
#add id to dateMac
df_dateMac = pd.read_csv(os.getcwd()+"/dacang/cleaned_data/dacang_dateMac_advanced_cleaned.csv")
df_dateMac['id'] = df_dateMac.oriMac.apply(AddId)
df_dateMac.head()

,M,Dur,A_count,Count,oriMac,date,Moment,id
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5,22.73,r
1,"7.5-20,178,229,137,47,142",17.02,4.0,38.0,"20,178,229,137,47,142",7.5,20.33,r
2,"7.6-4,207,140,11,128,186",23.98,4.0,947.0,"4,207,140,11,128,186",7.6,12.00,r
3,"7.6-208,151,254,40,222,169",23.97,4.0,910.0,"208,151,254,40,222,169",7.6,11.98,r
4,"7.5-48,74,38,133,29,213",17.00,5.0,49.0,"48,74,38,133,29,213",7.5,20.35,r


In [64]:
len(df_dateMac.oriMac.unique())

258

In [50]:
df_total = utils.Normalize_df(df_dateMac.reindex(columns = ['Dur','A_count','Count','Moment']))
df_total['id'] = df_dateMac.id
df_total['oriMac'] = df_dateMac.oriMac
df_total['m'] = df_dateMac.M
df_total.head()

,Dur,A_count,Count,Moment,id,oriMac,m
0,9.065624,1.818182,0.578723,9.575552,r,"48,74,38,155,214,98","7.5-48,74,38,155,214,98"
1,6.975228,0.000000,0.114894,8.556876,r,"20,178,229,137,47,142","7.5-20,178,229,137,47,142"
2,10.000000,0.000000,3.982979,5.021222,r,"4,207,140,11,128,186","7.6-4,207,140,11,128,186"
3,9.995654,0.000000,3.825532,5.012733,r,"208,151,254,40,222,169","7.6-208,151,254,40,222,169"
4,6.966536,0.909091,0.161702,8.565365,r,"48,74,38,133,29,213","7.5-48,74,38,133,29,213"


In [51]:
#整体分布情况
myplot.Scatter_3D(df_total,'Dur','Count','Moment','A_count',color_name='id')

In [53]:
#居民分布情况
df_resident = df_total[df_total.id == 'r']
print('居民dateMac数量:',len(df_resident))

居民dateMac数量: 774


In [65]:
dd = df_resident.groupby('oriMac').m.value_counts().to_frame().reset_index()

In [69]:
dd.oriMac.value_counts()[40:]

oriMac
40,53,69,96,185,108        9
24,2,174,207,62,51         9
64,140,31,52,191,11        9
196,64,246,220,81,179      9
104,74,174,67,128,132      7
112,58,81,118,159,27       7
128,214,5,75,8,213         7
156,116,3,130,126,7        6
20,154,16,154,204,112      6
8,250,121,183,224,101      6
124,218,195,126,234,181    6
208,151,254,41,36,162      6
48,148,53,197,58,225       6
48,123,172,71,160,113      6
128,138,139,124,94,97      5
116,21,117,93,235,62       5
140,170,206,161,68,84      5
220,245,5,135,187,95       5
192,46,37,42,205,205       4
180,71,245,166,44,220      4
56,251,20,50,26,70         4
112,138,9,131,175,199      4
8,250,121,194,19,245       4
88,214,151,126,55,223      3
20,178,229,137,47,142      3
220,109,205,125,41,209     3
48,161,250,41,151,15       3
28,209,7,152,33,91         3
32,50,51,210,96,61         2
Name: count, dtype: int64